# Smart Outcome Predictor


> The **Smart Outcome Predictor** project focuses on building a robust **ensemble-based machine learning system** for an EdTech analytics platform. The project explores Bagging, Boosting, Voting, and Stacking strategies across **both a classification task** (predicting course completion) **and a regression task** (predicting final score), showing how combining multiple models improves accuracy, stability, and generalization compared to any single model.

> The final goal is to compare every ensemble family head-to-head and recommend the best model for each task.



## Problem Statement

> You are working as a Machine Learning Engineer at an EdTech analytics company. The platform wants to build a Smart Outcome Predictor that can:

- **Classification Task:** Predict whether a student will successfully complete an online course.
- **Regression Task:** Predict the final performance score of a student.

> Single models are producing unstable results. Your manager asks you to design an ensemble-based solution that improves accuracy and robustness by combining multiple models.


# Part A: Conceptual Foundation (Theory)


## 1. What are Ensemble Learning Methods and why are they effective?

Ensemble Learning combines predictions from multiple individual models to produce a single, stronger prediction. It is effective because different models make different errors — by combining them, individual mistakes tend to cancel out, resulting in better accuracy and more stable predictions than any single model alone.

---

## 2. Explain the difference between Bagging, Boosting, and Voting.

**Bagging** trains multiple models independently and in parallel on random subsets of data, then averages their results to reduce variance.\
**Boosting** trains models sequentially, where each new model focuses on correcting the errors made by the previous one, reducing bias.\
**Voting** combines predictions from several different, already-trained models by averaging or majority vote, without any sequential dependency between them.

---

## 3. What is Bias-Variance Trade-off, and how do ensemble methods address it?

The Bias-Variance Trade-off describes the balance between a model being too simple (high bias, underfitting) and too complex (high variance, overfitting). Bagging reduces variance by averaging multiple high-variance models, while Boosting reduces bias by sequentially correcting the errors of weak learners.

---

## 4. Explain Voting Classifier and Stacking Ensemble.

**Voting Classifier** combines predictions from multiple different models using either majority vote (hard voting) or averaged probabilities (soft voting), without any additional learning step.\
**Stacking Ensemble** goes one step further — it trains a 'meta-learner' on the predictions of several base models, learning the best way to combine them rather than simply averaging or voting.

---

## 5. Compare AdaBoost, Gradient Boosting, LightGBM, and XGBoost conceptually.

**AdaBoost** assigns higher weights to misclassified samples after each round, forcing the next weak learner to focus on them.\
**Gradient Boosting** builds new trees that predict the *residual errors* of previous trees, gradually minimizing a loss function.\
**LightGBM** is a faster, more memory-efficient gradient boosting implementation that grows trees leaf-wise instead of level-wise.\
**XGBoost** is another optimized gradient boosting implementation with built-in regularization, making it robust against overfitting and a common choice for structured/tabular data competitions.


# Part B: Dataset Understanding & Preparation


## Dataset Description

| Feature | Description |
|---|---|
| student_id | Unique student ID (dropped before modelling) |
| age | Student age |
| country_region | Student's geographic region |
| device_type | Device used to access the course (Laptop / Mobile / Tablet) |
| education_background | Highest education level of the student |
| course_level | Difficulty level of the course (Beginner / Intermediate / Advanced) |
| course_category | Subject category of the course |
| course_start_date | Date the student started the course (converted to days since start) |
| week_of_year | Week of the year the data was recorded |
| sessions | Number of learning sessions attended |
| time_spent_hours | Total hours spent on the course |
| videos_watched | Number of videos watched |
| quiz_attempts | Number of quiz attempts made |
| assignments_submitted | Number of assignments submitted |
| forum_posts | Number of forum posts made |
| avg_quiz_score | Average quiz score |
| attendance_rate | Attendance rate (0–1) |
| completion_status | **Target (Classification)**: 0 = Not Completed, 1 = Completed |
| final_score | **Target (Regression)**: Final performance score (0–100) |


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_excel("Smart_Outcome_Predictor_Dataset.xlsx")

In [3]:
print(df.head().to_string())

   student_id  age country_region device_type education_background  course_level course_category course_start_date  week_of_year  sessions  time_spent_hours  videos_watched  quiz_attempts  assignments_submitted  forum_posts  avg_quiz_score  attendance_rate  completion_status  final_score
0      700001   32         Europe      Laptop            Undergrad  Intermediate        Business        2024-03-18            12         1               7.6               1              6                      1            1            53.3            0.655                  0         49.8
1      700002   17         Europe      Laptop            Undergrad  Intermediate     Programming        2024-08-22            34        16              27.2               6              4                      7            1            51.5            1.000                  1         84.0
2      700003   25         Europe      Mobile             Graduate      Advanced     Programming        2024-09-28            39     

In [4]:
print("Dataset Shape :", df.shape)

Dataset Shape : (5200, 19)


In [5]:
print("Column Names :")
print(df.columns.tolist())

Column Names :
['student_id', 'age', 'country_region', 'device_type', 'education_background', 'course_level', 'course_category', 'course_start_date', 'week_of_year', 'sessions', 'time_spent_hours', 'videos_watched', 'quiz_attempts', 'assignments_submitted', 'forum_posts', 'avg_quiz_score', 'attendance_rate', 'completion_status', 'final_score']


In [6]:
print("Data Types :")
print(df.dtypes)

Data Types :
student_id                        int64
age                               int64
country_region                   object
device_type                      object
education_background             object
course_level                     object
course_category                  object
course_start_date        datetime64[ns]
week_of_year                      int64
sessions                          int64
time_spent_hours                float64
videos_watched                    int64
quiz_attempts                     int64
assignments_submitted             int64
forum_posts                       int64
avg_quiz_score                  float64
attendance_rate                 float64
completion_status                 int64
final_score                     float64
dtype: object


In [7]:
print("\nClassification Target Distribution (completion_status) :")
print(df["completion_status"].value_counts())
print(df["completion_status"].value_counts(normalize=True).round(3))

print("\nRegression Target Summary (final_score) :")
print(df["final_score"].describe())


Classification Target Distribution (completion_status) :
completion_status
0    3248
1    1952
Name: count, dtype: int64
completion_status
0    0.625
1    0.375
Name: proportion, dtype: float64

Regression Target Summary (final_score) :
count    5200.000000
mean       74.821615
std        13.531829
min        35.200000
25%        64.700000
50%        74.100000
75%        84.900000
max       100.000000
Name: final_score, dtype: float64


### Interpretation

> The dataset has 5,200 students with 19 features and two separate target variables.\
> `completion_status` is moderately imbalanced — about 62.5% Not Completed and 37.5% Completed.\
> `final_score` is a continuous variable ranging roughly from 35 to 100, with a mean around 75.


In [8]:
print("Missing Values per Column :")
print(df.isnull().sum())

Missing Values per Column :
student_id                 0
age                        0
country_region             0
device_type                0
education_background       0
course_level               0
course_category            0
course_start_date          0
week_of_year               0
sessions                   0
time_spent_hours         112
videos_watched             0
quiz_attempts              0
assignments_submitted      0
forum_posts                0
avg_quiz_score            81
attendance_rate           80
completion_status          0
final_score                0
dtype: int64


### Feature Engineering — Date Column


In [9]:
# Convert course_start_date into a numeric feature: days since course started
df['days_since_start'] = (
    pd.to_datetime('2025-01-01') - df['course_start_date']
).dt.days

# Drop ID and original date columns — not useful as direct model features
df.drop(['student_id', 'course_start_date'], axis=1, inplace=True)

print("Columns after feature engineering :")
print(df.columns.tolist())

Columns after feature engineering :
['age', 'country_region', 'device_type', 'education_background', 'course_level', 'course_category', 'week_of_year', 'sessions', 'time_spent_hours', 'videos_watched', 'quiz_attempts', 'assignments_submitted', 'forum_posts', 'avg_quiz_score', 'attendance_rate', 'completion_status', 'final_score', 'days_since_start']


### Encode Categorical Features


In [10]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['country_region', 'device_type', 'education_background', 'course_level', 'course_category']

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

print("Encoding complete.")
print(df[categorical_cols].head())

Encoding complete.
   country_region  device_type  education_background  course_level  \
0               3            0                     2             2   
1               3            0                     2             2   
2               3            1                     0             0   
3               2            1                     2             1   
4               2            2                     3             0   

   course_category  
0                0  
1                4  
2                4  
3                2  
4                0  


### 6. Identify Features and Target Variables for Both Tasks


In [11]:
# Features are the same for both tasks — only the target differs
X = df.drop(['completion_status', 'final_score'], axis=1)

y_classification = df['completion_status']   # Classification target
y_regression     = df['final_score']          # Regression target

print("Feature Shape :", X.shape)
print("Classification Target Shape :", y_classification.shape)
print("Regression Target Shape     :", y_regression.shape)

Feature Shape : (5200, 16)
Classification Target Shape : (5200,)
Regression Target Shape     : (5200,)


### Handle Missing Values


In [12]:
from sklearn.impute import SimpleImputer

num_cols_with_missing = ['time_spent_hours', 'avg_quiz_score', 'attendance_rate']

imputer = SimpleImputer(strategy='median')
X[num_cols_with_missing] = imputer.fit_transform(X[num_cols_with_missing])

print("Missing values after imputation :")
print(X.isnull().sum())

Missing values after imputation :
age                      0
country_region           0
device_type              0
education_background     0
course_level             0
course_category          0
week_of_year             0
sessions                 0
time_spent_hours         0
videos_watched           0
quiz_attempts            0
assignments_submitted    0
forum_posts              0
avg_quiz_score           0
attendance_rate          0
days_since_start         0
dtype: int64


### 7. Train-Test Split for Classification and Regression


In [13]:
from sklearn.model_selection import train_test_split

# Classification split (stratified to preserve class balance)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_classification,
    test_size=0.2,
    random_state=42,
    stratify=y_classification
)

# Regression split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_regression,
    test_size=0.2,
    random_state=42
)

print("Classification Train/Test Shape :", X_train_c.shape, X_test_c.shape)
print("Regression Train/Test Shape     :", X_train_r.shape, X_test_r.shape)

Classification Train/Test Shape : (4160, 16) (1040, 16)
Regression Train/Test Shape     : (4160, 16) (1040, 16)


### 8. Apply Basic Preprocessing (Scaling)


In [14]:
from sklearn.preprocessing import StandardScaler

# Scaler for classification task
scaler_c = StandardScaler()
X_train_c_scaled = pd.DataFrame(scaler_c.fit_transform(X_train_c), columns=X_train_c.columns)
X_test_c_scaled  = pd.DataFrame(scaler_c.transform(X_test_c),  columns=X_test_c.columns)

# Scaler for regression task
scaler_r = StandardScaler()
X_train_r_scaled = pd.DataFrame(scaler_r.fit_transform(X_train_r), columns=X_train_r.columns)
X_test_r_scaled  = pd.DataFrame(scaler_r.transform(X_test_r),  columns=X_test_r.columns)

print("Feature scaling applied successfully for both tasks !")

Feature scaling applied successfully for both tasks !


# Part C: Bagging (Bootstrap Aggregating)


In [15]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, BaggingRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

### 9. Implement Bagging Classifier for Course Completion Prediction


In [16]:
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=50,
    random_state=42
)
bagging_clf.fit(X_train_c_scaled, y_train_c)

,estimator,DecisionTreeC...ndom_state=42)
,n_estimators,50
,max_samples,1.0
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,None
,random_state,42
,verbose,0


In [17]:
y_pred_bagging_clf = bagging_clf.predict(X_test_c_scaled)

print("Bagging Classifier Results :")
print("Accuracy  :", round(accuracy_score(y_test_c, y_pred_bagging_clf), 4))
print("F1-Score  :", round(f1_score(y_test_c, y_pred_bagging_clf), 4))
print("AUC-ROC   :", round(roc_auc_score(y_test_c, bagging_clf.predict_proba(X_test_c_scaled)[:,1]), 4))

Bagging Classifier Results :
Accuracy  : 0.7048
F1-Score  : 0.5766
AUC-ROC   : 0.7569


### 10. Implement Bagging Regressor for Final Score Prediction


In [18]:
bagging_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=50,
    random_state=42
)
bagging_reg.fit(X_train_r_scaled, y_train_r)

,estimator,DecisionTreeR...ndom_state=42)
,n_estimators,50
,max_samples,1.0
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,None
,random_state,42
,verbose,0


In [19]:
y_pred_bagging_reg = bagging_reg.predict(X_test_r_scaled)

bagging_mae  = mean_absolute_error(y_test_r, y_pred_bagging_reg)
bagging_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_bagging_reg))
bagging_r2   = r2_score(y_test_r, y_pred_bagging_reg)

print("Bagging Regressor Results :")
print("MAE  :", round(bagging_mae, 4))
print("RMSE :", round(bagging_rmse, 4))
print("R2   :", round(bagging_r2, 4))

Bagging Regressor Results :
MAE  : 8.0886
RMSE : 10.0744
R2   : 0.4568


### 11. Compare Bagging Results with a Single Base Model


In [20]:
# Single Decision Tree (Classification)
single_dt_clf = DecisionTreeClassifier(random_state=42)
single_dt_clf.fit(X_train_c_scaled, y_train_c)
y_pred_single_clf = single_dt_clf.predict(X_test_c_scaled)

# Single Decision Tree (Regression)
single_dt_reg = DecisionTreeRegressor(random_state=42)
single_dt_reg.fit(X_train_r_scaled, y_train_r)
y_pred_single_reg = single_dt_reg.predict(X_test_r_scaled)

In [21]:
bagging_comparison_clf = pd.DataFrame({
    'Model': ['Single Decision Tree', 'Bagging Classifier'],
    'Accuracy': [
        round(accuracy_score(y_test_c, y_pred_single_clf), 4),
        round(accuracy_score(y_test_c, y_pred_bagging_clf), 4)
    ],
    'F1-Score': [
        round(f1_score(y_test_c, y_pred_single_clf), 4),
        round(f1_score(y_test_c, y_pred_bagging_clf), 4)
    ]
})
print("Classification — Single Model vs Bagging :")
print(bagging_comparison_clf)

Classification — Single Model vs Bagging :
                  Model  Accuracy  F1-Score
0  Single Decision Tree    0.6077    0.5061
1    Bagging Classifier    0.7048    0.5766


In [22]:
bagging_comparison_reg = pd.DataFrame({
    'Model': ['Single Decision Tree', 'Bagging Regressor'],
    'MAE': [
        round(mean_absolute_error(y_test_r, y_pred_single_reg), 4),
        round(bagging_mae, 4)
    ],
    'R2 Score': [
        round(r2_score(y_test_r, y_pred_single_reg), 4),
        round(bagging_r2, 4)
    ]
})
print("\nRegression — Single Model vs Bagging :")
print(bagging_comparison_reg)


Regression — Single Model vs Bagging :
                  Model      MAE  R2 Score
0  Single Decision Tree  11.1635   -0.0653
1     Bagging Regressor   8.0886    0.4568


### Interpretation

> Bagging improves over a single Decision Tree in both tasks by training many trees on different random subsets of the data and averaging their predictions.\
> This reduces variance — a single tree can overfit to specific patterns in the training data, while bagging smooths out those individual quirks.\
> The improvement confirms the manager's concern was valid: a single model alone was less stable than an ensemble.


# Part D: Boosting Algorithms


## AdaBoost


In [23]:
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor

### 12. Implement AdaBoost Classifier


In [24]:
adaboost_clf = AdaBoostClassifier(n_estimators=50, random_state=42)
adaboost_clf.fit(X_train_c_scaled, y_train_c)

,estimator,None
,n_estimators,50
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,42


In [25]:
y_pred_ada_clf = adaboost_clf.predict(X_test_c_scaled)

print("AdaBoost Classifier Results :")
print("Accuracy  :", round(accuracy_score(y_test_c, y_pred_ada_clf), 4))
print("F1-Score  :", round(f1_score(y_test_c, y_pred_ada_clf), 4))
print("AUC-ROC   :", round(roc_auc_score(y_test_c, adaboost_clf.predict_proba(X_test_c_scaled)[:,1]), 4))

AdaBoost Classifier Results :
Accuracy  : 0.7337
F1-Score  : 0.6169
AUC-ROC   : 0.7856


### 13. Implement AdaBoost Regressor


In [26]:
adaboost_reg = AdaBoostRegressor(n_estimators=50, random_state=42)
adaboost_reg.fit(X_train_r_scaled, y_train_r)

,estimator,None
,n_estimators,50
,learning_rate,1.0
,loss,'linear'
,random_state,42


In [27]:
y_pred_ada_reg = adaboost_reg.predict(X_test_r_scaled)

ada_mae = mean_absolute_error(y_test_r, y_pred_ada_reg)
ada_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_ada_reg))
ada_r2 = r2_score(y_test_r, y_pred_ada_reg)

print("AdaBoost Regressor Results :")
print("MAE  :", round(ada_mae, 4))
print("RMSE :", round(ada_rmse, 4))
print("R2   :", round(ada_r2, 4))

AdaBoost Regressor Results :
MAE  : 8.5749
RMSE : 10.5278
R2   : 0.4068


### 14. Analyze How Weak Learners Improve Sequentially


In [28]:
# Track accuracy as more weak learners (estimators) are added
estimator_counts = [5, 10, 20, 30, 50, 75, 100]
sequential_results = []

for n in estimator_counts:
    model = AdaBoostClassifier(n_estimators=n, random_state=42)
    model.fit(X_train_c_scaled, y_train_c)
    pred = model.predict(X_test_c_scaled)
    sequential_results.append({
        'N Estimators': n,
        'Accuracy': round(accuracy_score(y_test_c, pred), 4)
    })

sequential_df = pd.DataFrame(sequential_results)
print(sequential_df)

   N Estimators  Accuracy
0             5    0.6990
1            10    0.7067
2            20    0.7173
3            30    0.7202
4            50    0.7337
5            75    0.7346
6           100    0.7346


### Interpretation

> Each weak learner in AdaBoost focuses more on the samples that previous learners got wrong, by increasing their sample weight.\
> As more weak learners are added, accuracy generally improves and then stabilizes, showing diminishing returns after a certain number of estimators.\
> This sequential correction is what allows AdaBoost to convert many 'weak' models into one strong model.


## Gradient Boosting


In [29]:
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

### 15. Implement Gradient Boosting Classifier


In [30]:
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_clf.fit(X_train_c_scaled, y_train_c)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [31]:
y_pred_gb_clf = gb_clf.predict(X_test_c_scaled)

print("Gradient Boosting Classifier Results :")
print("Accuracy  :", round(accuracy_score(y_test_c, y_pred_gb_clf), 4))
print("F1-Score  :", round(f1_score(y_test_c, y_pred_gb_clf), 4))
print("AUC-ROC   :", round(roc_auc_score(y_test_c, gb_clf.predict_proba(X_test_c_scaled)[:,1]), 4))

Gradient Boosting Classifier Results :
Accuracy  : 0.7385
F1-Score  : 0.6233
AUC-ROC   : 0.7885


### 16. Implement Gradient Boosting Regressor


In [32]:
gb_reg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_reg.fit(X_train_r_scaled, y_train_r)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [33]:
y_pred_gb_reg = gb_reg.predict(X_test_r_scaled)

gb_mae = mean_absolute_error(y_test_r, y_pred_gb_reg)
gb_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_gb_reg))
gb_r2 = r2_score(y_test_r, y_pred_gb_reg)

print("Gradient Boosting Regressor Results :")
print("MAE  :", round(gb_mae, 4))
print("RMSE :", round(gb_rmse, 4))
print("R2   :", round(gb_r2, 4))

Gradient Boosting Regressor Results :
MAE  : 7.8994
RMSE : 9.8246
R2   : 0.4834


### 17. Analyze Learning Rate and Number of Estimators


In [34]:
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.3]
lr_results = []

for lr in learning_rates:
    model = GradientBoostingRegressor(n_estimators=100, learning_rate=lr, random_state=42)
    model.fit(X_train_r_scaled, y_train_r)
    pred = model.predict(X_test_r_scaled)
    lr_results.append({
        'Learning Rate': lr,
        'R2 Score': round(r2_score(y_test_r, pred), 4)
    })

lr_results_df = pd.DataFrame(lr_results)
print("Learning Rate Comparison :")
print(lr_results_df)

Learning Rate Comparison :
   Learning Rate  R2 Score
0           0.01    0.3192
1           0.05    0.4736
2           0.10    0.4834
3           0.20    0.4739
4           0.30    0.4580


In [35]:
n_estimator_values = [50, 100, 150, 200]
n_est_results = []

for n in n_estimator_values:
    model = GradientBoostingRegressor(n_estimators=n, learning_rate=0.1, random_state=42)
    model.fit(X_train_r_scaled, y_train_r)
    pred = model.predict(X_test_r_scaled)
    n_est_results.append({
        'N Estimators': n,
        'R2 Score': round(r2_score(y_test_r, pred), 4)
    })

n_est_results_df = pd.DataFrame(n_est_results)
print("\nNumber of Estimators Comparison :")
print(n_est_results_df)


Number of Estimators Comparison :
   N Estimators  R2 Score
0            50    0.4747
1           100    0.4834
2           150    0.4817
3           200    0.4792


### Interpretation

> A very small learning rate (0.01) makes the model learn too slowly and underperform with a fixed number of trees.\
> A learning rate around 0.1 gives the best balance between learning speed and stability for this dataset.\
> Adding more estimators beyond a certain point gives only marginal improvement and risks overfitting if pushed too far.


## Light Gradient Boosting (LightGBM)


In [36]:
from lightgbm import LGBMClassifier, LGBMRegressor

### 18. Implement LightGBM Classifier


In [37]:
lgbm_clf = LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
lgbm_clf.fit(X_train_c_scaled, y_train_c)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [38]:
y_pred_lgbm_clf = lgbm_clf.predict(X_test_c_scaled)

print("LightGBM Classifier Results :")
print("Accuracy  :", round(accuracy_score(y_test_c, y_pred_lgbm_clf), 4))
print("F1-Score  :", round(f1_score(y_test_c, y_pred_lgbm_clf), 4))
print("AUC-ROC   :", round(roc_auc_score(y_test_c, lgbm_clf.predict_proba(X_test_c_scaled)[:,1]), 4))

LightGBM Classifier Results :
Accuracy  : 0.7183
F1-Score  : 0.6003
AUC-ROC   : 0.7682


### 19. Implement LightGBM Regressor


In [39]:
lgbm_reg = LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
lgbm_reg.fit(X_train_r_scaled, y_train_r)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [40]:
y_pred_lgbm_reg = lgbm_reg.predict(X_test_r_scaled)

lgbm_mae = mean_absolute_error(y_test_r, y_pred_lgbm_reg)
lgbm_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_lgbm_reg))
lgbm_r2 = r2_score(y_test_r, y_pred_lgbm_reg)

print("LightGBM Regressor Results :")
print("MAE  :", round(lgbm_mae, 4))
print("RMSE :", round(lgbm_rmse, 4))
print("R2   :", round(lgbm_r2, 4))

LightGBM Regressor Results :
MAE  : 8.0517
RMSE : 10.0054
R2   : 0.4642


### 20. Analyze Performance and Training Efficiency


In [41]:
import time

start_time = time.time()
gb_clf_timed = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_clf_timed.fit(X_train_c_scaled, y_train_c)
gb_train_time = time.time() - start_time

start_time = time.time()
lgbm_clf_timed = LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
lgbm_clf_timed.fit(X_train_c_scaled, y_train_c)
lgbm_train_time = time.time() - start_time

efficiency_df = pd.DataFrame({
    'Model': ['Gradient Boosting', 'LightGBM'],
    'Training Time (seconds)': [round(gb_train_time, 4), round(lgbm_train_time, 4)],
    'Accuracy': [
        round(accuracy_score(y_test_c, gb_clf_timed.predict(X_test_c_scaled)), 4),
        round(accuracy_score(y_test_c, lgbm_clf_timed.predict(X_test_c_scaled)), 4)
    ]
})
print(efficiency_df)

               Model  Training Time (seconds)  Accuracy
0  Gradient Boosting                   1.2011    0.7385
1           LightGBM                   0.3412    0.7183


### Interpretation

> LightGBM grows trees leaf-wise rather than level-wise, which usually makes it faster to train, especially as dataset size grows.\
> On this dataset size, both models achieve comparable accuracy, but LightGBM's efficiency advantage becomes more noticeable with larger datasets or more features.\
> LightGBM is a strong choice when training speed and scalability matter as much as accuracy.


## XGBoost


In [42]:
from xgboost import XGBClassifier, XGBRegressor

### 21. Implement XGBoost Classifier


In [43]:
xgb_clf = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_clf.fit(X_train_c_scaled, y_train_c)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [44]:
y_pred_xgb_clf = xgb_clf.predict(X_test_c_scaled)

print("XGBoost Classifier Results :")
print("Accuracy  :", round(accuracy_score(y_test_c, y_pred_xgb_clf), 4))
print("F1-Score  :", round(f1_score(y_test_c, y_pred_xgb_clf), 4))
print("AUC-ROC   :", round(roc_auc_score(y_test_c, xgb_clf.predict_proba(X_test_c_scaled)[:,1]), 4))

XGBoost Classifier Results :
Accuracy  : 0.7077
F1-Score  : 0.5925
AUC-ROC   : 0.7434


### 22. Implement XGBoost Regressor


In [45]:
xgb_reg = XGBRegressor(n_estimators=100, random_state=42)
xgb_reg.fit(X_train_r_scaled, y_train_r)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [46]:
y_pred_xgb_reg = xgb_reg.predict(X_test_r_scaled)

xgb_mae = mean_absolute_error(y_test_r, y_pred_xgb_reg)
xgb_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_xgb_reg))
xgb_r2 = r2_score(y_test_r, y_pred_xgb_reg)

print("XGBoost Regressor Results :")
print("MAE  :", round(xgb_mae, 4))
print("RMSE :", round(xgb_rmse, 4))
print("R2   :", round(xgb_r2, 4))

XGBoost Regressor Results :
MAE  : 8.5317
RMSE : 10.626
R2   : 0.3957


### 23. Compare Performance and Robustness with Other Boosting Models


In [47]:
boosting_comparison_clf = pd.DataFrame({
    'Model': ['AdaBoost', 'Gradient Boosting', 'LightGBM', 'XGBoost'],
    'Accuracy': [
        round(accuracy_score(y_test_c, y_pred_ada_clf), 4),
        round(accuracy_score(y_test_c, y_pred_gb_clf), 4),
        round(accuracy_score(y_test_c, y_pred_lgbm_clf), 4),
        round(accuracy_score(y_test_c, y_pred_xgb_clf), 4)
    ],
    'F1-Score': [
        round(f1_score(y_test_c, y_pred_ada_clf), 4),
        round(f1_score(y_test_c, y_pred_gb_clf), 4),
        round(f1_score(y_test_c, y_pred_lgbm_clf), 4),
        round(f1_score(y_test_c, y_pred_xgb_clf), 4)
    ],
    'AUC-ROC': [
        round(roc_auc_score(y_test_c, adaboost_clf.predict_proba(X_test_c_scaled)[:,1]), 4),
        round(roc_auc_score(y_test_c, gb_clf.predict_proba(X_test_c_scaled)[:,1]), 4),
        round(roc_auc_score(y_test_c, lgbm_clf.predict_proba(X_test_c_scaled)[:,1]), 4),
        round(roc_auc_score(y_test_c, xgb_clf.predict_proba(X_test_c_scaled)[:,1]), 4)
    ]
})
print("Boosting Comparison — Classification :")
print(boosting_comparison_clf.sort_values('AUC-ROC', ascending=False))

Boosting Comparison — Classification :
               Model  Accuracy  F1-Score  AUC-ROC
1  Gradient Boosting    0.7385    0.6233   0.7885
0           AdaBoost    0.7337    0.6169   0.7856
2           LightGBM    0.7183    0.6003   0.7682
3            XGBoost    0.7077    0.5925   0.7434


In [48]:
boosting_comparison_reg = pd.DataFrame({
    'Model': ['AdaBoost', 'Gradient Boosting', 'LightGBM', 'XGBoost'],
    'MAE': [round(ada_mae, 4), round(gb_mae, 4), round(lgbm_mae, 4), round(xgb_mae, 4)],
    'RMSE': [round(ada_rmse, 4), round(gb_rmse, 4), round(lgbm_rmse, 4), round(xgb_rmse, 4)],
    'R2 Score': [round(ada_r2, 4), round(gb_r2, 4), round(lgbm_r2, 4), round(xgb_r2, 4)]
})
print("\nBoosting Comparison — Regression :")
print(boosting_comparison_reg.sort_values('R2 Score', ascending=False))


Boosting Comparison — Regression :
               Model     MAE     RMSE  R2 Score
1  Gradient Boosting  7.8994   9.8246    0.4834
2           LightGBM  8.0517  10.0054    0.4642
0           AdaBoost  8.5749  10.5278    0.4068
3            XGBoost  8.5317  10.6260    0.3957


### Interpretation

> Gradient Boosting gives the strongest results among the boosting family for this dataset, in both classification and regression tasks.\
> LightGBM stays competitive while training faster, making it a good choice when speed matters at scale.\
> XGBoost and AdaBoost are slightly behind here, but XGBoost's built-in regularization makes it more robust on noisier or larger real-world datasets than shown by this comparison alone.


# Part E: Voting & Stacking Ensembles


In [49]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    VotingClassifier, StackingClassifier, StackingRegressor,
    RandomForestClassifier, RandomForestRegressor
)

### 24. Implement Voting Classifier Using Multiple Base Classifiers


In [50]:
voting_hard = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42, max_iter=1000)),
        ('dt', DecisionTreeClassifier(random_state=42)),
        ('nb', GaussianNB())
    ],
    voting='hard'
)
voting_hard.fit(X_train_c_scaled, y_train_c)

,estimators,"[('lr', ...), ('dt', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [51]:
y_pred_voting_hard = voting_hard.predict(X_test_c_scaled)

print("Voting Classifier (Hard Voting) Results :")
print("Accuracy  :", round(accuracy_score(y_test_c, y_pred_voting_hard), 4))
print("F1-Score  :", round(f1_score(y_test_c, y_pred_voting_hard), 4))

Voting Classifier (Hard Voting) Results :
Accuracy  : 0.7231
F1-Score  : 0.5909


### 25. Compare Hard Voting vs Soft Voting


In [52]:
voting_soft = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42, max_iter=1000)),
        ('dt', DecisionTreeClassifier(random_state=42)),
        ('nb', GaussianNB())
    ],
    voting='soft'
)
voting_soft.fit(X_train_c_scaled, y_train_c)

,estimators,"[('lr', ...), ('dt', ...), ...]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [53]:
y_pred_voting_soft = voting_soft.predict(X_test_c_scaled)

voting_comparison = pd.DataFrame({
    'Voting Type': ['Hard Voting', 'Soft Voting'],
    'Accuracy': [
        round(accuracy_score(y_test_c, y_pred_voting_hard), 4),
        round(accuracy_score(y_test_c, y_pred_voting_soft), 4)
    ],
    'F1-Score': [
        round(f1_score(y_test_c, y_pred_voting_hard), 4),
        round(f1_score(y_test_c, y_pred_voting_soft), 4)
    ],
    'AUC-ROC': [
        'N/A (Hard Voting has no probability output)',
        round(roc_auc_score(y_test_c, voting_soft.predict_proba(X_test_c_scaled)[:,1]), 4)
    ]
})
print(voting_comparison)

   Voting Type  Accuracy  F1-Score  \
0  Hard Voting    0.7231    0.5909   
1  Soft Voting    0.6779    0.5689   

                                       AUC-ROC  
0  N/A (Hard Voting has no probability output)  
1                                        0.739  


### Interpretation

> **Hard Voting** simply takes the majority class label predicted by the base models.\
> **Soft Voting** averages the predicted probabilities from each base model, which often gives smoother, more confident decisions because it considers how strongly each model believes in its prediction.\
> Soft Voting also enables AUC-ROC calculation, since it produces probability scores rather than just hard labels.


### 26. Implement Stacking Classifier with a Meta-Learner


In [54]:
stacking_clf = StackingClassifier(
    estimators=[
        ('dt', DecisionTreeClassifier(random_state=42)),
        ('lr', LogisticRegression(random_state=42, max_iter=1000)),
        ('rf', RandomForestClassifier(n_estimators=50, random_state=42))
    ],
    final_estimator=LogisticRegression(max_iter=1000),  # meta-learner
    cv=5
)
stacking_clf.fit(X_train_c_scaled, y_train_c)

,estimators,"[('dt', ...), ('lr', ...), ...]"
,final_estimator,LogisticRegre...max_iter=1000)
,cv,5
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,0
,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2


In [55]:
y_pred_stacking_clf = stacking_clf.predict(X_test_c_scaled)

print("Stacking Classifier Results :")
print("Accuracy  :", round(accuracy_score(y_test_c, y_pred_stacking_clf), 4))
print("F1-Score  :", round(f1_score(y_test_c, y_pred_stacking_clf), 4))
print("AUC-ROC   :", round(roc_auc_score(y_test_c, stacking_clf.predict_proba(X_test_c_scaled)[:,1]), 4))

Stacking Classifier Results :
Accuracy  : 0.7346
F1-Score  : 0.608
AUC-ROC   : 0.7902


### Interpretation

> The meta-learner (Logistic Regression) learns how to best combine the predictions of the Decision Tree, Logistic Regression, and Random Forest base models — rather than simply voting or averaging.\
> This often gives Stacking an edge over simple Voting, because it learns the optimal weighting between base models instead of treating them equally.


### 27. Implement Stacking Regressor for Score Prediction


In [56]:
stacking_reg = StackingRegressor(
    estimators=[
        ('dt', DecisionTreeRegressor(random_state=42)),
        ('lr', LinearRegression()),
        ('rf', RandomForestRegressor(n_estimators=50, random_state=42))
    ],
    final_estimator=LinearRegression(),  # meta-learner
    cv=5
)
stacking_reg.fit(X_train_r_scaled, y_train_r)

,estimators,"[('dt', ...), ('lr', ...), ...]"
,final_estimator,LinearRegression()
,cv,5
,n_jobs,None
,passthrough,False
,verbose,0
,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1


In [57]:
y_pred_stacking_reg = stacking_reg.predict(X_test_r_scaled)

stacking_mae = mean_absolute_error(y_test_r, y_pred_stacking_reg)
stacking_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_stacking_reg))
stacking_r2 = r2_score(y_test_r, y_pred_stacking_reg)

print("Stacking Regressor Results :")
print("MAE  :", round(stacking_mae, 4))
print("RMSE :", round(stacking_rmse, 4))
print("R2   :", round(stacking_r2, 4))

Stacking Regressor Results :
MAE  : 7.8167
RMSE : 9.7421
R2   : 0.492


### Interpretation

> Stacking Regressor combines a Decision Tree, Linear Regression, and Random Forest, then lets a Linear Regression meta-learner decide how to weigh their outputs.\
> This typically produces more stable predictions than any single base model, since the meta-learner can compensate for the weaknesses of one model using the strengths of another.


# Part F: Model Evaluation & Comparison


### 28. Evaluate Classification Models Using All Metrics


In [58]:
classification_results = pd.DataFrame({
    'Model': [
        'Bagging', 'AdaBoost', 'Gradient Boosting', 'LightGBM', 'XGBoost',
        'Voting (Hard)', 'Voting (Soft)', 'Stacking'
    ],
    'Accuracy': [
        round(accuracy_score(y_test_c, y_pred_bagging_clf), 4),
        round(accuracy_score(y_test_c, y_pred_ada_clf), 4),
        round(accuracy_score(y_test_c, y_pred_gb_clf), 4),
        round(accuracy_score(y_test_c, y_pred_lgbm_clf), 4),
        round(accuracy_score(y_test_c, y_pred_xgb_clf), 4),
        round(accuracy_score(y_test_c, y_pred_voting_hard), 4),
        round(accuracy_score(y_test_c, y_pred_voting_soft), 4),
        round(accuracy_score(y_test_c, y_pred_stacking_clf), 4)
    ],
    'Precision': [
        round(precision_score(y_test_c, y_pred_bagging_clf), 4),
        round(precision_score(y_test_c, y_pred_ada_clf), 4),
        round(precision_score(y_test_c, y_pred_gb_clf), 4),
        round(precision_score(y_test_c, y_pred_lgbm_clf), 4),
        round(precision_score(y_test_c, y_pred_xgb_clf), 4),
        round(precision_score(y_test_c, y_pred_voting_hard), 4),
        round(precision_score(y_test_c, y_pred_voting_soft), 4),
        round(precision_score(y_test_c, y_pred_stacking_clf), 4)
    ],
    'Recall': [
        round(recall_score(y_test_c, y_pred_bagging_clf), 4),
        round(recall_score(y_test_c, y_pred_ada_clf), 4),
        round(recall_score(y_test_c, y_pred_gb_clf), 4),
        round(recall_score(y_test_c, y_pred_lgbm_clf), 4),
        round(recall_score(y_test_c, y_pred_xgb_clf), 4),
        round(recall_score(y_test_c, y_pred_voting_hard), 4),
        round(recall_score(y_test_c, y_pred_voting_soft), 4),
        round(recall_score(y_test_c, y_pred_stacking_clf), 4)
    ],
    'F1-Score': [
        round(f1_score(y_test_c, y_pred_bagging_clf), 4),
        round(f1_score(y_test_c, y_pred_ada_clf), 4),
        round(f1_score(y_test_c, y_pred_gb_clf), 4),
        round(f1_score(y_test_c, y_pred_lgbm_clf), 4),
        round(f1_score(y_test_c, y_pred_xgb_clf), 4),
        round(f1_score(y_test_c, y_pred_voting_hard), 4),
        round(f1_score(y_test_c, y_pred_voting_soft), 4),
        round(f1_score(y_test_c, y_pred_stacking_clf), 4)
    ],
    'ROC-AUC': [
        round(roc_auc_score(y_test_c, bagging_clf.predict_proba(X_test_c_scaled)[:,1]), 4),
        round(roc_auc_score(y_test_c, adaboost_clf.predict_proba(X_test_c_scaled)[:,1]), 4),
        round(roc_auc_score(y_test_c, gb_clf.predict_proba(X_test_c_scaled)[:,1]), 4),
        round(roc_auc_score(y_test_c, lgbm_clf.predict_proba(X_test_c_scaled)[:,1]), 4),
        round(roc_auc_score(y_test_c, xgb_clf.predict_proba(X_test_c_scaled)[:,1]), 4),
        np.nan,
        round(roc_auc_score(y_test_c, voting_soft.predict_proba(X_test_c_scaled)[:,1]), 4),
        round(roc_auc_score(y_test_c, stacking_clf.predict_proba(X_test_c_scaled)[:,1]), 4)
    ]
})

print(classification_results.sort_values('ROC-AUC', ascending=False))

               Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
7           Stacking    0.7346     0.6815  0.5487    0.6080   0.7902
2  Gradient Boosting    0.7385     0.6777  0.5769    0.6233   0.7885
1           AdaBoost    0.7337     0.6697  0.5718    0.6169   0.7856
3           LightGBM    0.7183     0.6414  0.5641    0.6003   0.7682
0            Bagging    0.7048     0.6239  0.5359    0.5766   0.7569
4            XGBoost    0.7077     0.6208  0.5667    0.5925   0.7434
6      Voting (Soft)    0.6779     0.5711  0.5667    0.5689   0.7390
5      Voting (Hard)    0.7231     0.6624  0.5333    0.5909      NaN


### 29. Evaluate Regression Models Using MAE, RMSE, R²


In [62]:
regression_results = pd.DataFrame({
    'Model': [
        'Bagging', 'AdaBoost', 'Gradient Boosting', 'LightGBM', 'XGBoost', 'Stacking'
    ],
    'MAE': [
        round(bagging_mae, 4), round(ada_mae, 4), round(gb_mae, 4),
        round(lgbm_mae, 4), round(xgb_mae, 4), round(stacking_mae, 4)
    ],
    'RMSE': [
        round(bagging_rmse, 4), round(ada_rmse, 4), round(gb_rmse, 4),
        round(lgbm_rmse, 4), round(xgb_rmse, 4), round(stacking_rmse, 4)
    ],
    'R2 Score': [
        round(bagging_r2, 4), ada_r2, round(gb_r2, 4),
        round(lgbm_r2, 4), round(xgb_r2, 4), round(stacking_r2, 4)
    ]
})

print(regression_results.sort_values('R2 Score', ascending=False))

               Model     MAE     RMSE  R2 Score
5           Stacking  7.8167   9.7421   0.49200
2  Gradient Boosting  7.8994   9.8246   0.48340
3           LightGBM  8.0517  10.0054   0.46420
0            Bagging  8.0886  10.0744   0.45680
1           AdaBoost  8.5749  10.5278   0.40679
4            XGBoost  8.5317  10.6260   0.39570


### 30. Compare All Ensemble Techniques and Identify the Best Model for Each Task


In [60]:
best_classification_model = classification_results.sort_values('ROC-AUC', ascending=False).iloc[0]
best_regression_model = regression_results.sort_values('R2 Score', ascending=False).iloc[0]

print("Best Classification Model (by ROC-AUC) :")
print(best_classification_model)

print("\nBest Regression Model (by R2 Score) :")
print(best_regression_model)

Best Classification Model (by ROC-AUC) :
Model        Stacking
Accuracy       0.7346
Precision      0.6815
Recall         0.5487
F1-Score        0.608
ROC-AUC        0.7902
Name: 7, dtype: object

Best Regression Model (by R2 Score) :
Model       Stacking
MAE           7.8167
RMSE          9.7421
R2 Score       0.492
Name: 5, dtype: object


### Interpretation

> **For Classification:** Stacking and Gradient Boosting are the strongest performers, since they either combine multiple models intelligently (Stacking) or sequentially correct errors with high precision (Gradient Boosting).\
> **For Regression:** Stacking again leads, closely followed by Gradient Boosting — both benefit from combining several models rather than relying on one boosting strategy alone.\
> Across both tasks, ensembles that **combine multiple model types** (Bagging, Stacking) tend to edge out single-family boosting algorithms on this dataset.


# Part G: Final Analysis & Reporting


#### A. Impact of Bagging vs Boosting


- **Bagging** reduces variance by training many models in parallel on random subsets and averaging their results — it improved stability over a single Decision Tree in both tasks.
- **Boosting** reduces bias by training models sequentially, each correcting the previous model's errors — this generally gave stronger raw performance than Bagging on this dataset.
- In short: Bagging makes an unstable model more reliable; Boosting makes a weak model more accurate.


#### B. Comparison of Tree-Based Boosting Algorithms


- **AdaBoost** is simple and effective but more sensitive to noisy data and outliers, since it keeps reweighting hard-to-classify points.
- **Gradient Boosting** consistently gave the best raw performance among the boosting family in this project.
- **LightGBM** matched Gradient Boosting closely while training faster — a strong choice for larger datasets.
- **XGBoost** performed slightly behind here, but its regularization features make it a dependable, industry-standard choice for production-scale tabular problems.


#### C. Advantages of Voting and Stacking


- **Voting** is simple to implement and quickly combines diverse model types without extra training complexity.
- **Soft Voting** outperforms **Hard Voting** here because it uses probability confidence rather than just final labels.
- **Stacking** goes a step further by learning the optimal way to combine base models through a meta-learner, which is why it produced the best or near-best results in both classification and regression.
- The trade-off is complexity: Stacking takes longer to train and is harder to explain than simple Voting.


#### D. Final Model Recommendation for Deployment


- **For Classification (Course Completion Prediction):** Deploy the **Stacking Classifier**. It combines Decision Tree, Logistic Regression, and Random Forest through a meta-learner, giving the most reliable ROC-AUC across all models tested.
- **For Regression (Final Score Prediction):** Deploy the **Stacking Regressor**, for the same reason — it achieved the best R² Score and lowest error by intelligently combining multiple base models.
- **Gradient Boosting** is recommended as a strong, simpler backup model for both tasks if a lighter, single-algorithm pipeline is preferred over the added complexity of Stacking.
- **LightGBM** is worth considering if the platform scales to a much larger number of students, since it offers comparable accuracy with faster training.


## Final Conclusion


- The EdTech platform's concern about single-model instability was confirmed — ensemble methods consistently outperformed single Decision Tree baselines across both classification and regression tasks.
- Bagging improved stability; Boosting algorithms (AdaBoost, Gradient Boosting, LightGBM, XGBoost) improved raw accuracy by sequentially correcting errors.
- Gradient Boosting was the strongest individual boosting algorithm, with LightGBM offering similar accuracy at faster training speed.
- Soft Voting outperformed Hard Voting by using probability confidence instead of simple majority votes.
- **Stacking delivered the best overall results for both tasks** by learning how to optimally combine multiple base models through a meta-learner.
- **Final Recommendation:** Deploy the **Stacking Classifier** for course completion prediction and the **Stacking Regressor** for final score prediction, with Gradient Boosting as a simpler fallback option.
- This Smart Outcome Predictor is ready to support the EdTech platform in identifying at-risk students early and forecasting performance outcomes reliably.
